## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-26-22-49-27 +0000,bbc,US too focused on Middle East and not pushing ...,https://www.bbc.co.uk/news/articles/c820q9wgrp...
1,2026-11-26-22-41-05 +0000,nypost,Indiana perv walks free on $2K bail after ‘fil...,https://nypost.com/2026/11/26/us-news/indiana-...
2,2026-11-26-22-35-33 +0000,startribune,Dozens displaced on Thanksgiving after St. Pau...,https://www.startribune.com/fire-st-paul-weste...
3,2026-11-26-22-32-58 +0000,nypost,Mamdani sent in NYPD’s SRG to quell anti-Israe...,https://nypost.com/2026/11/26/us-news/mamdani-...
4,2026-11-26-22-23-01 +0000,cbc,Non-permanent population expected to exceed 20...,https://www.cbc.ca/news/politics/temporary-imm...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2720/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
27,thanksgiving,29
342,trump,18
166,he,15
63,new,13
660,china,11


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
129,2026-11-26-14-47-00 +0000,wsj,Macy’s Postpones Thanksgiving Day Parade as Po...,https://www.wsj.com/us-news/macys-postpones-it...,75
40,2026-11-26-20-37-19 +0000,nypost,Trump celebrates Thanksgiving with his holiday...,https://nypost.com/2026/11/26/us-news/trump-ce...,74
48,2026-11-26-20-18-20 +0000,nypost,Rescheduled Macy’s Thanksgiving Day Parade set...,https://nypost.com/2026/11/26/us-news/reschedu...,71
115,2026-11-26-15-39-23 +0000,nypost,Trump tells Putin to ‘pull back all military a...,https://nypost.com/2026/11/26/us-news/trump-te...,69
223,2026-11-26-10-00-00 +0000,wapo,"Which came first, Macy’s Thanksgiving parade o...",https://www.washingtonpost.com/nation/interact...,66


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
129,75,2026-11-26-14-47-00 +0000,wsj,Macy’s Postpones Thanksgiving Day Parade as Po...,https://www.wsj.com/us-news/macys-postpones-it...
115,69,2026-11-26-15-39-23 +0000,nypost,Trump tells Putin to ‘pull back all military a...,https://nypost.com/2026/11/26/us-news/trump-te...
92,48,2026-11-26-17-12-53 +0000,wapo,Two Italians killed in Kuwait as a strained Ir...,https://www.washingtonpost.com/world/2026/11/2...
201,42,2026-11-26-10-25-45 +0000,nypost,Russia targets Kyiv with ballistic missiles — ...,https://nypost.com/2026/11/26/world-news/russi...
45,39,2026-11-26-20-30-14 +0000,nypost,Nvidia halts server shipments to 4 countries j...,https://nypost.com/2026/11/26/business/nvidia-...
179,38,2026-11-26-11-21-45 +0000,bbc,Drivers over 70 face eye tests every three yea...,https://www.bbc.co.uk/news/articles/cjqlqerzv1...
159,35,2026-11-26-12-42-00 +0000,wsj,A Game-Changing Move Just Gave America the Edg...,https://www.wsj.com/tech/a-game-changing-move-...
205,35,2026-11-26-10-13-24 +0000,nypost,Hong Kong marks anniversary high-rise blaze th...,https://nypost.com/2026/11/26/world-news/hong-...
220,33,2026-11-26-10-00-00 +0000,wsj,Democratic Sen. Chris Van Hollen wants to be S...,https://www.wsj.com/politics/policy/why-this-d...
31,33,2026-11-26-20-58-12 +0000,nypost,Owner of Jewish NYC deli targeted by anti-Isra...,https://nypost.com/2026/11/26/us-news/owner-of...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
